In [10]:
import pandas as pd 
from sklearn.tree import DecisionTreeRegressor
import kagglehub
from sklearn.model_selection import train_test_split

path = kagglehub.dataset_download("dansbecker/melbourne-housing-snapshot")
home_data = pd.read_csv(path + "/melb_data.csv")
home_data = home_data.dropna(axis=0)


home_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 
                        'YearBuilt', 'Lattitude', 'Longtitude']

X = home_data[home_features]
y = home_data.Price

train_X, val_X, train_y, val_y = train_test_split(X, y,random_state = 0)
# train x is the training features, val x is the validation features,
#  train y is the training prices, and val y is the validation prices.

From what I understand Random Forst is just desecian trees that trained with different training examples it helps for overfitting mostly and if there are more 

In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

forest_model = RandomForestRegressor(random_state=1,
                                      n_estimators=20, # that gives us 20 trees in the forest as u tried 
                                      # it's not better to have more in the forest all the time default 100
                                      max_depth=50, # sane as descision tree, just for each tree default is None,
                                    # which means nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.
                                      min_samples_split=10 # if we have 10 samples in a node, default is 2 
                                       # we will not split it anymore, just for each tree u can use for desicion tree as well
                                     )
forest_model.fit(train_X, train_y)
melb_preds = forest_model.predict(val_X)
print(mean_absolute_error(val_y, melb_preds))

198283.0626426113


best desicion tree has given 274054 but even without changing hyper parameters random forest gives 191669

In [12]:
def train_try_model_all_data(n_estimators,max_depth,min_samples_split,train_X,train_y):
    result_dict = {}
    for i in n_estimators:
        for j in max_depth:
            for z in min_samples_split:
                model = RandomForestRegressor(
                    n_estimators = i,
                    max_depth = j,
                    min_samples_split = z
                )
    
                model.fit(train_X,train_y)
                predictions = model.predict(val_X)
                
                mae = mean_absolute_error(predictions,val_y)
                result_dict[(i, j, z)] = mae
    return result_dict

In [16]:
results = train_try_model_all_data(
    n_estimators = [10, 20, 30],
    max_depth = [10, 20, 30],
    min_samples_split = [2, 5, 10],
    train_X = train_X,
    train_y = train_y
)

best_parameters, best_mae = min(
    results.items(),
    key=lambda item: item[1]
)
print(f"Best parameters: n_estimators={best_parameters[0]}, max_depth={best_parameters[1]}, min_samples_split={best_parameters[2] } with MAE={best_mae}")

Best parameters: n_estimators=20, max_depth=20, min_samples_split=5 with MAE=191763.44495618515
